# Gamma vs ZAGamma: NBA PTS Diagnostic
Train NBA PTS with plain Gamma (no zero-inflation) and compare against the current ZAGamma results. Quick hyperparameter search to keep iteration fast.

In [1]:
import pandas as pd
import numpy as np
import json
import importlib.resources as pkg_resources
from sportstradamus import data
from sportstradamus.helpers import get_odds, get_ev, set_model_start_values, stat_cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, accuracy_score, log_loss
from scipy.stats import gamma as gamma_dist
from scipy.optimize import minimize
import lightgbm as lgb
from lightgbmlss.model import LightGBMLSS
from lightgbmlss.distributions.Gamma import Gamma
from lightgbmlss.distributions.ZAGamma import ZAGamma
import warnings
warnings.filterwarnings('ignore')
np.random.seed(69)

/home/trevor/Sportstradamus/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Training Data and Feature Columns

In [2]:
# Load feature filter to get the right columns
with open(pkg_resources.files(data) / "feature_filter.json") as f:
    feature_filter = json.load(f)

league, market = "NBA", "PTS"
cols = (feature_filter.get("Common", [])
        + feature_filter[league]["Common"]
        + feature_filter[league][market])

profile_cols = [c for c in feature_filter[league][market]
                if "Player " in c and not any(s in c for s in [" age", " depth", " proj "])]

count = 1
for i, c in enumerate(cols.copy()):
    if c in profile_cols:
        cols.insert(i + count, c + " growth")
        cols.insert(i + count, c + " short")
        count += 2

# Load training data
filepath = pkg_resources.files(data) / "training_data/NBA_PTS.csv"
M = pd.read_csv(filepath, index_col=0)
print(f"Training data: {len(M)} rows, {len(cols)} feature columns")
print(f"Zero results: {(M.Result == 0).sum()} / {len(M)} = {(M.Result == 0).mean():.4f}")
print(f"Result stats: mean={M.Result.mean():.2f}, std={M.Result.std():.2f}")

Training data: 46000 rows, 91 feature columns
Zero results: 1390 / 46000 = 0.0302
Result stats: mean=13.80, std=8.78


## Prepare Train / Validation / Test Split

In [3]:
y = M[['Result']]
X = M[cols]

categories = ["Home", "Player position"]
if "Player position" not in X.columns:
    categories.remove("Player position")
for c in categories:
    X[c] = X[c].astype('category')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=25)

B_test = M.loc[X_test.index, ['Line', 'Odds', 'EV']]
B_val = M.loc[X_val.index, ['Line', 'Odds', 'EV']]

y_train_labels = np.ravel(y_train.to_numpy())
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 32200, Val: 6900, Test: 6900


## Train: Plain Gamma (no zero-inflation)

In [4]:
dist = "Gamma"
dist_obj = Gamma(stabilization="None", loss_fn="nll", response_fn="softplus")
model_gamma = LightGBMLSS(dist_obj)

# Gamma requires y > 0. Filter out zeros entirely rather than patching
# (eps replacement causes alpha<1 exploitation → -inf NLL).
nonzero_mask = y_train_labels > 0
X_train_g = X_train[nonzero_mask]
y_train_g = y_train_labels[nonzero_mask]
print(f"Dropped {(~nonzero_mask).sum()} zeros → {len(y_train_g)} rows for Gamma training")

# Don't use per-obs start values — library CV code assumes scalar start_values
# (gradient_fn takes [0,:] of init_score, breaking per-obs values).
# Let the library compute its own scalar start values from the non-zero labels.
set_model_start_values(model_gamma, dist, X_train_g)
dtrain = lgb.Dataset(X_train_g, label=y_train_g)

# Quick hyper_opt: fewer trials, shorter time budget
params = {
    "feature_pre_filter": ["none", [False]],
    "num_threads": ["none", [8]],
    "max_depth": ["int", {"low": 4, "high": 12, "log": False}],
    "max_bin": ["none", [63]],
    "num_leaves": ["int", {"low": 23, "high": 128, "log": False}],
    "lambda_l1": ["float", {"low": 1e-4, "high": 10, "log": True}],
    "lambda_l2": ["float", {"low": 1e-4, "high": 10, "log": True}],
    "min_child_samples": ["int", {"low": 30, "high": 100, "log": False}],
    "min_child_weight": ["float", {"low": 1e-3, "high": .75*len(X_train_g)/1000, "log": True}],
    "learning_rate": ["float", {"low": 0.005, "high": 0.1, "log": True}],
    "feature_fraction": ["float", {"low": 0.4, "high": 1.0, "log": False}],
    "bagging_fraction": ["float", {"low": 0.4, "high": 1.0, "log": False}],
    "bagging_freq": ["none", [1]]
}
opt_params_gamma = model_gamma.hyper_opt(
    params, dtrain,
    num_boost_round=500,
    nfold=4,
    early_stopping_rounds=50,
    max_minutes=10,
    n_trials=50,
    silence=True,
)
print(f"Gamma opt_rounds: {opt_params_gamma['opt_rounds']}")

Dropped 976 zeros → 31224 rows for Gamma training


Best trial: 0. Best value: 3.24917:   4%|▍         | 2/50 [00:40<16:21, 20.44s/it, 40.72/600 seconds]


[W 2026-03-11 22:08:31,782] Trial 2 failed with parameters: {'feature_pre_filter': False, 'num_threads': 8, 'max_depth': 9, 'max_bin': 63, 'num_leaves': 99, 'lambda_l1': 0.0038018482666847342, 'lambda_l2': 0.0022434185978291252, 'min_child_samples': 31, 'min_child_weight': 0.14435782908611908, 'learning_rate': 0.005401549296173468, 'feature_fraction': 0.6348480298563128, 'bagging_fraction': 0.9997950088165612, 'bagging_freq': 1, 'boosting': 'gbdt'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/trevor/Sportstradamus/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/trevor/Sportstradamus/.venv/lib/python3.11/site-packages/lightgbmlss/model.py", line 366, in objective
    lgblss_param_tuning = self.cv(hyper_params,
                          ^^^^^^^^^^^^^^^^^^^^^
  File "/home/trevor/Sportstradamus/.venv/lib/python3.1

KeyboardInterrupt: 

In [ ]:
# Train final Gamma model on non-zero data
model_gamma.train(opt_params_gamma, dtrain, num_boost_round=opt_params_gamma["opt_rounds"])

# Predict on FULL test set (including zeros) — use library's scalar start_values
preds_gamma = model_gamma.predict(X_test, pred_type="parameters")
preds_gamma.index = X_test.index

# Predict on validation set
preds_gamma_val = model_gamma.predict(X_val, pred_type="parameters")
preds_gamma_val.index = X_val.index

print("Gamma test predictions:")
print(preds_gamma.describe())

## Gamma: Sanity Checks

In [ ]:
alpha_g = preds_gamma["concentration"].to_numpy()
beta_g = preds_gamma["rate"].to_numpy()
ev_g = alpha_g / beta_g
result = y_test["Result"].to_numpy()

# CDF of actual results under model's Gamma
cdf = gamma_dist.cdf(result, alpha_g, scale=1/beta_g)
print("=== Gamma CDF of actual results ===")
print(f"  mean={cdf.mean():.4f}, std={cdf.std():.4f}")
print(f"  < 0.05: {(cdf < 0.05).mean():.4f}")
print(f"  > 0.95: {(cdf > 0.95).mean():.4f}")
print(f"  in [0.2, 0.8]: {((cdf > 0.2) & (cdf < 0.8)).mean():.4f}")
print()
print("=== Gamma parameter stats ===")
print(f"  alpha: mean={alpha_g.mean():.2f}, median={np.median(alpha_g):.2f}, max={alpha_g.max():.2f}")
print(f"  beta: mean={beta_g.mean():.4f}, median={np.median(beta_g):.4f}")
print(f"  ev (alpha/beta): mean={ev_g.mean():.2f}, std={ev_g.std():.2f}")
print(f"  effective std (ev/sqrt(alpha)): mean={(ev_g/np.sqrt(alpha_g)).mean():.2f}")
print(f"  actual result: mean={result.mean():.2f}, std={result.std():.2f}")

## Gamma: Accuracy Metrics (no book blending)

In [ ]:
# Raw model odds (no book blending, no calibration)
lines = B_test["Line"].to_numpy()
step = M["Result"].drop_duplicates().sort_values().diff().min()

y_proba_raw = get_odds(lines, ev_g, "Gamma", alpha=alpha_g, step=step)
y_proba = np.array([y_proba_raw, 1 - y_proba_raw]).T

y_class = (y_test["Result"].to_numpy() >= lines).astype(int)
y_pred = (y_proba > 0.5).astype(int)[:, 1]

mask = np.max(y_proba, axis=1) > 0.54
print("=== Raw Gamma (no blending) ===")
print(f"  Accuracy (all): {accuracy_score(y_class, y_pred):.4f}")
print(f"  Accuracy (confident): {accuracy_score(y_class[mask], y_pred[mask]):.4f}")
print(f"  Precision (confident): {precision_score(y_class[mask], y_pred[mask]):.4f}")
print(f"  Sharpness: {np.std(y_proba[:, 1]):.4f}")
print(f"  Log-loss: {log_loss(y_class, y_proba[:, 1]):.4f}")
print(f"  Confident fraction: {mask.mean():.4f}")

## Train: ZAGamma (zero-adjusted) for comparison

In [8]:
dist_za = "ZAGamma"
hist_gate = 0.098  # from stat_zi.json
dist_obj_za = ZAGamma(stabilization="None", loss_fn="nll", response_fn="softplus")
model_za = LightGBMLSS(dist_obj_za)
# set_model_start_values(model_za, dist_za, X_train, hist_gate=hist_gate)

# ZAGamma uses the ORIGINAL labels (with actual zeros)
dtrain_za = lgb.Dataset(X_train, label=y_train_labels)

opt_params_za = model_za.hyper_opt(
    params, dtrain_za,
    num_boost_round=500,
    nfold=4,
    early_stopping_rounds=50,
    max_minutes=10,
    n_trials=50,
    silence=True,
)
print(f"ZAGamma opt_rounds: {opt_params_za['opt_rounds']}")

Best trial: 0. Best value: 4.74789:   2%|▏         | 1/50 [00:42<34:23, 42.12s/it, 15.15/600 seconds]

[W 2026-03-11 21:25:43,537] Trial 1 failed with parameters: {'feature_pre_filter': False, 'num_threads': 8, 'max_depth': 9, 'max_bin': 63, 'num_leaves': 25, 'lambda_l1': 0.00022505345382502597, 'lambda_l2': 0.42323656476216936, 'min_child_samples': 94, 'min_child_weight': 0.5344746099429909, 'learning_rate': 0.007508628634033074, 'feature_fraction': 0.6546565338582149, 'bagging_fraction': 0.856570276247224, 'bagging_freq': 1, 'boosting': 'gbdt'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/trevor/Sportstradamus/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/trevor/Sportstradamus/.venv/lib/python3.11/site-packages/lightgbmlss/model.py", line 366, in objective
    lgblss_param_tuning = self.cv(hyper_params,
                          ^^^^^^^^^^^^^^^^^^^^^
  File "/home/trevor/Sportstradamus/.venv/lib/python3.11/s

KeyboardInterrupt: 

In [ ]:
# Train final ZAGamma model
model_za.train(opt_params_za, dtrain_za, num_boost_round=opt_params_za["opt_rounds"])

# Predict on test set
set_model_start_values(model_za, dist_za, X_test, hist_gate=hist_gate)
preds_za = model_za.predict(X_test, pred_type="parameters")
preds_za.index = X_test.index

print("ZAGamma test predictions:")
print(preds_za.describe())

## ZAGamma: Sanity Checks

In [ ]:
alpha_za = preds_za["concentration"].to_numpy()
beta_za = preds_za["rate"].to_numpy()
gate_za = preds_za["gate"].to_numpy()
ev_za = alpha_za / beta_za

# CDF of actual results under model's base Gamma (ignoring gate)
cdf_za = gamma_dist.cdf(result, alpha_za, scale=1/beta_za)
print("=== ZAGamma base Gamma CDF of actual results ===")
print(f"  mean={cdf_za.mean():.4f}, std={cdf_za.std():.4f}")
print(f"  < 0.05: {(cdf_za < 0.05).mean():.4f}")
print(f"  > 0.95: {(cdf_za > 0.95).mean():.4f}")
print()
print("=== ZAGamma parameter stats ===")
print(f"  alpha: mean={alpha_za.mean():.2f}, median={np.median(alpha_za):.2f}, max={alpha_za.max():.2f}")
print(f"  beta: mean={beta_za.mean():.4f}, median={np.median(beta_za):.4f}")
print(f"  gate: mean={gate_za.mean():.4f}, std={gate_za.std():.4f}")
print(f"  ev (alpha/beta): mean={ev_za.mean():.2f}, std={ev_za.std():.2f}")
print(f"  effective std (ev/sqrt(alpha)): mean={(ev_za/np.sqrt(alpha_za)).mean():.2f}")

In [ ]:
# ZAGamma raw model odds
y_proba_za_raw = get_odds(lines, ev_za, "ZAGamma", alpha=alpha_za, step=step, gate=gate_za)
y_proba_za = np.array([y_proba_za_raw, 1 - y_proba_za_raw]).T

y_pred_za = (y_proba_za > 0.5).astype(int)[:, 1]
mask_za = np.max(y_proba_za, axis=1) > 0.54

print("=== Raw ZAGamma (no blending) ===")
print(f"  Accuracy (all): {accuracy_score(y_class, y_pred_za):.4f}")
print(f"  Accuracy (confident): {accuracy_score(y_class[mask_za], y_pred_za[mask_za]):.4f}")
print(f"  Precision (confident): {precision_score(y_class[mask_za], y_pred_za[mask_za]):.4f}")
print(f"  Sharpness: {np.std(y_proba_za[:, 1]):.4f}")
print(f"  Log-loss: {log_loss(y_class, y_proba_za[:, 1]):.4f}")
print(f"  Confident fraction: {mask_za.mean():.4f}")

## Side-by-Side Comparison

In [ ]:
print(f"{'Metric':<25} {'Gamma':>10} {'ZAGamma':>10}")
print("-" * 47)
print(f"{'Alpha mean':<25} {alpha_g.mean():>10.2f} {alpha_za.mean():>10.2f}")
print(f"{'Alpha median':<25} {np.median(alpha_g):>10.2f} {np.median(alpha_za):>10.2f}")
print(f"{'Alpha max':<25} {alpha_g.max():>10.2f} {alpha_za.max():>10.2f}")
print(f"{'EV mean':<25} {ev_g.mean():>10.2f} {ev_za.mean():>10.2f}")
print(f"{'Eff. std mean':<25} {(ev_g/np.sqrt(alpha_g)).mean():>10.2f} {(ev_za/np.sqrt(alpha_za)).mean():>10.2f}")
print(f"{'Gate mean':<25} {'N/A':>10} {gate_za.mean():>10.4f}")
print(f"{'CDF mean':<25} {gamma_dist.cdf(result, alpha_g, scale=1/beta_g).mean():>10.4f} {cdf_za.mean():>10.4f}")
print(f"{'CDF > 0.95 frac':<25} {(gamma_dist.cdf(result, alpha_g, scale=1/beta_g) > 0.95).mean():>10.4f} {(cdf_za > 0.95).mean():>10.4f}")
print(f"{'Sharpness':<25} {np.std(y_proba[:, 1]):>10.4f} {np.std(y_proba_za[:, 1]):>10.4f}")
print(f"{'Log-loss':<25} {log_loss(y_class, y_proba[:, 1]):>10.4f} {log_loss(y_class, y_proba_za[:, 1]):>10.4f}")
print(f"{'Accuracy (confident)':<25} {accuracy_score(y_class[mask], y_pred[mask]):>10.4f} {accuracy_score(y_class[mask_za], y_pred_za[mask_za]):>10.4f}")